In [ ]:
%load_ext dotenv
%dotenv config.env

In [ ]:
%load_ext autoreload 
%autoreload 2

In [ ]:
import random
import h5py
import importlib
import seaborn as sns
import subprocess
import uuid
import trackpy as tp
import numpy as np
import pandas as pd
import os
import pickle
from matplotlib import pyplot as plt
from pathlib import Path
from ipyfilechooser import FileChooser
from utils.conditions_to_dict import conditions_to_df, get_fovs
from ipywidgets import interactive, fixed
from utils.set_widgets import filter_data, h5_w_tracking_params, manual_reg_widgets, fov_selector

from utils.conditions_to_dict import conditions_to_df, get_fovs
from utils.logging import setup_logger, log_and_print, save_as_pickle
from utils.set_widgets import *
from utils.plot import *
from utils.tiff_writer import download_img_allas, show_mask, fluo_as_array, coords_to_image
from utils.locs_trackpy import get_locs_trackpy, h5_get_locs_preview
from utils.track_trackpy import track_py, plot_tracks, locs_preview_contours
from skimage import measure
from utils.csv_writer import show_df_style
from io import BytesIO
from utils.dnaK_proj import bandpass
from utils.dnaK_proj import detect_aggregates_in_cells
from utils.dnaK_proj import get_dist_to_agg
from shapely import points, distance

In [ ]:
# Select the aoutput directory to save plots, logs and data 

path = Path(os.getenv('SMPP_INPUT_DIR'))
output_for_data = FileChooser(path)
output_for_data.show_only_dirs = True
display(output_for_data)


In [ ]:
# setup logging

analysis_id = str(uuid.uuid1())[:8]
git_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode('ascii').strip()

logger = setup_logger(output_for_data, analysis_id)
log_and_print(f'output directory: {output_for_data.selected_path}')
log_and_print(f'analysis_id_short:{analysis_id}')
log_and_print(f'git_hash:{git_hash}')


In [ ]:
# Click 'Play' to search across the database and choose the experiment which you'd like to analyse.
# This script will only work with the experiments performed on PyME gui after 27.02.2025

# Only one experiment entry is allowed to be processed.
# Use the 'clear' function in the 'date' field to remove any unwanted entry.
# It is ok to use a comma and space to separate the row numbers in the 'exclude' and 'include' fields.
# Keep all the fields intact to view the entire database (though this may become inconvenient soon)

database = pd.read_csv(os.getenv('SMPP_DATABASE'), sep = '\t', dtype = 'str')
fields_list, fld_widgets_list, box = filter_data(database)
display(box)


In [ ]:
# Run this cell when ready with the selection above,
# check if the experiment you choose is the one you want.

experiment = conditions_to_df(database, fld_widgets_list, fields_list, 'Experiments')
experiment


In [ ]:
# Run this cell to see selected imaging files

exp_id = experiment['uuid_experiment'].values[0]
files, masks = get_fovs(database, exp_id)

log_and_print('\nSelected files:')
show_df_style(files)
log_and_print('\n\nSelected masks:')
show_df_style(masks)
log_and_print('')


In [ ]:
# Select the FOVs you'd like to process (must have a mask associated, see 'masks:' field)

box, output, checked_ids = fov_all_selector(files, masks)
log_and_print('')
display(box, output)
log_and_print('')


In [ ]:
# Show selected FOVs

files_selected = files[files['uuid_fov'].isin(checked_ids)]
masks_selected = masks[masks['uuid_fov'].isin(checked_ids)]

log_and_print(f"\n\n{files_selected.shape[0]} files selected, {masks_selected.shape[0]} masks\n")
show_df_style(files_selected)
log_and_print('')


In [ ]:
# Get the tracking data from Allas (only the tracking channel for each FOV will be shown)
# (might take few min)

master_im_data = []
master_im_data_2ch = []
master_fluos = []
master_fluos_2ch = []

for file in files_selected['uuid_file'].unique():
    path = eval(files.loc[files['uuid_file']==file, 'path'].item())
    h5 = h5py.File(BytesIO(download_img_allas(path[0], path[1])), 'r')
    log_and_print(f"{path[1]} downloaded")
    
    if h5['ImageData'].shape[0] > 10**4:

        
        try:
            fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
            p_mask = masks.loc[masks['uuid_fov']==fov_id, 'path'].item()
            mask_uuid = masks.loc[masks['uuid_fov']==fov_id, 'uuid_analysis'].item()
            
            master_im_data.append((file, path, p_mask, mask_uuid))
            master_fluos.append(h5['ImageData'])

            m_fname = p_mask.split('/')[-1]
            m_dir   = '/'.join(p_mask.split('/')[:-1])
            m = show_mask(m_dir, m_fname, imshow=False)
            proj = np.mean(h5['ImageData'][0:42,:,:], axis=0)

            fig, axs = plt.subplots(1, 2)
            axs[0].imshow(proj)
            axs[1].imshow(m)
            plt.show()
            plt.close()
            
            log_and_print(f"FOV: {fov_id}")
            log_and_print(f"{path[1]} added to fluos (tracking)")
            print('\n================================================\n\n')
            
        except ValueError:
            log_and_print(f'can\'t find mask for file {path[1]}')

    elif h5['MetaData'].attrs['LEDpower']==0: # not widefield
           
        try:
            fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
            p_mask = masks.loc[masks['uuid_fov']==fov_id, 'path'].item()
            mask_uuid = masks.loc[masks['uuid_fov']==fov_id, 'uuid_analysis'].item()
            
            master_im_data_2ch.append((file, path, p_mask, mask_uuid))
            master_fluos_2ch.append(h5['ImageData'])

            m_fname = p_mask.split('/')[-1]
            m_dir   = '/'.join(p_mask.split('/')[:-1])
            m = show_mask(m_dir, m_fname, imshow=False)
            proj = np.mean(h5['ImageData'][0:42,:,:], axis=0)

            fig, axs = plt.subplots(1, 2)
            axs[0].imshow(proj)
            axs[1].imshow(m)
            plt.show()
            plt.close()

            log_and_print(f"FOV: {fov_id}")
            log_and_print(f"{path[1]} added to 2nd_channel ")
            print('\n================================================\n\n')
            
        except ValueError:
            log_and_print(f'can\'t find mask for file {path[1]}')
    
    else: 
        plt.imshow(np.mean(h5['ImageData'][0:42,:,:], axis=0))
        plt.show()
        plt.close()

        fov_id = files.loc[files['uuid_file']==file, 'uuid_fov'].item()
        log_and_print(f"FOV: {fov_id}")
        log_and_print(f"{path[1]} is brightfield")
        print('\n================================================\n\n')
        

print('All files are ready')



### Get number of cells with aggregate


In [ ]:
# This cell will generate the mean projections


images = []

for file, info in zip(master_fluos_2ch, master_im_data_2ch):
    print(file.shape)
    proj = np.mean(file[1:,:,:], axis = 0)
    plt.imshow(proj)
    plt.show()

    images.append(proj)


In [ ]:
# - diameter: approximate spot diameter in pixels (odd integer)
# - noise_size: sigma for small Gaussian (in pixels)
# - smoothing_size: window size for background (in pixels)

# --------detection of foci/aggregates using trackpy--------------

aggregates_ls = []
cell_masks = []
coords = []


for i in range(len(images)):
    p_mask = master_im_data_2ch[i][2]
    m_fname = p_mask.split('/')[-1]
    m_dir   = '/'.join(p_mask.split('/')[:-1])
    mask = show_mask(m_dir, m_fname, imshow=False)
    cell_masks.append(mask)

diameter = 7

for image, cell_mask in zip(images, cell_masks):
    feats, counts_per_cell, summary = detect_aggregates_in_cells(
        image, cell_mask,
        diameter=diameter,         # choose based on expected aggregate size
        minmass=None,        
        noise_size=1,
        smoothing_size=9,
        threshold=50,
        preprocess = False
    )
    
    plt.figure(figsize=(6,5))
    plt.imshow(image, cmap='gray')
    if len(feats) > 0:
        plt.scatter(feats['x'], feats['y'], s=30, facecolors='none', edgecolors='r')
    plt.title(f"Detected spots: {len(feats)}")
    plt.show()
    aggregates_ls.append([feats, counts_per_cell, summary])
    coords.append(feats)




In [ ]:
# Extract the dict from each [df, dict] pair
# ----Inspect no. of aggregates -----

records = [item[2] for item in aggregates_ls]  # assuming dict is at position 1
aggregate_stats = pd.DataFrame(records)
print(aggregate_stats)



In [ ]:
# ---------per-cell counts of reference points/aggregates------
# This assumes each row in coords is a reference point with columns ['x','y','cell_id_int'].
results = []
for dots in coords:
    dots = dots.copy()
    ref_counts_per_cell = (
        dots.groupby('cell_id_int')[['x','y']]
        .size()
        .rename('n_refs')
        .reset_index()
    )
    
    # Categorize into 1, 2, >=3
    ref_counts_per_cell['ref_cat'] = pd.cut(
        ref_counts_per_cell['n_refs'],
        bins=[0, 1, 2, np.inf],
        labels=['1', '2', '>=3'],
        right=True,
        include_lowest=True
    )
    
    # Summary: number of cells in each category (wide format)
    ref_counts_summary = (
        ref_counts_per_cell
        .groupby('ref_cat', dropna=False)
        .size()
        .reindex(['1','2','>=3'], fill_value=0)
        .rename('n_cells')
        .reset_index()
    )
    ref_counts_summary_wide = (
        ref_counts_summary
        .set_index('ref_cat')
        .T
        .rename_axis(None, axis=1)
    )
    # Rename columns to be explicit
    ref_counts_summary_wide = ref_counts_summary_wide.rename(columns={
        '1': 'n_cells_1ref', #No. of cells with 1/2 or more than 3 aggregates
        '2': 'n_cells_2refs',
        '>=3': 'n_cells_ge3refs'
    })

    results.append(ref_counts_summary_wide)


ref_data = pd.concat(results)
print(ref_data)

In [ ]:
# Dump the data
im_path = "folder_path"
filename = "filename_here"
aggregate_stats.to_csv(f'{im_path}/{filename}DotStats_size_filtered{exp_id[:8]}.csv')
ref_data.to_csv(f'{im_path}/{filename}DotsPerCell_size_filtered{exp_id[:8]}.csv')

print('Data saved')

In [ ]:
# read tracking files (.csv) to get track coordinates
file_path = "your_file_path"
tracks = []
paths = sorted(Path(file_path).glob('*.csv'))
for path in paths:
    df = pd.read_csv(path)
    tracks.append(df)



In [ ]:
result_ls = []

for locs, dots, mask in zip (tracks, coords, cell_masks):
    result = get_dist_to_agg(locs, dots, mask, show_plots= False)
    result_ls.append(result)

#Aggregate results in one dataframe
agg = pd.concat(result_ls)


In [ ]:
# Dump the data
filename = "your_filename"
ref_data.to_csv(f'{folder_path}/{filename}aggregateCounts_size_filtered{exp_id[:8]}.csv')
aggregate_stats.to_csv(f'{folder_path}/{filename}DotStats_size_filtered{exp_id[:8]}.csv')

print('Data saved')

In [ ]:
# -------Visualize distance to nearest aggregate state histograms for one experiment------
slow = agg[agg["Dapp_log"]<= -0.34]

plt.hist(slow["nearest_ref_dist_nm"], bins = 40 , density = True)
plt.show()

transient = agg[(agg["Dapp_log"] >= -0.34) & (agg["Dapp_log"] <= 0.74)]

plt.hist(transient["nearest_ref_dist_nm"], bins = 40 , density = True)
plt.show()

fast = agg[agg["Dapp_log"]>= 0.74] 

plt.hist(fast["nearest_ref_dist_nm"], bins = 40 , density = True)
plt.show()


Plot all experiments of one condition together with SEM

This section plots the two OE strains together in the same plots with their own SEMs

In [ ]:
# --- read files containing distance to aggregate (column= "nearest_ref_dist_nm") ---
directory = "your_folder_path"
dots_all = []
from pathlib import Path
paths = sorted(Path(directory).glob('*.csv'))
for path in paths:
    df = pd.read_csv(path)
    dots_all.append(df)
    


In [ ]:
#Plot state histograms of distance to aggregate for all OE data
label = ["FumA", "MetE"]
for i, data in enumerate(dots_all, start=1):
    # Split by state
    slow = data.loc[data["Dapp_log"] <= -0.34, "nearest_ref_dist_nm"].dropna()
    transient = data.loc[(data["Dapp_log"] > -0.34) & (data["Dapp_log"] <= 0.74), "nearest_ref_dist_nm"].dropna()
    fast = data.loc[data["Dapp_log"] > 0.36, "nearest_ref_dist_nm"].dropna()

    # One figure per dataset/iteration
    fig, ax = plt.subplots(figsize=(6, 4))

    # Consistent bins across the three groups for this dataset
    all_vals = pd.concat([slow, transient, fast])
    edges = np.histogram_bin_edges(all_vals, bins=30)

    # Three histograms on the same axes
    ax.hist(slow,      bins=edges, density=True, color='C0', alpha=0.45, label='slow', edgecolor = "k", histtype ="stepfilled")
    ax.hist(transient, bins=edges, density=True, color='C1', alpha=0.45, label='transient', edgecolor = "k", histtype ="stepfilled")
    ax.hist(fast,      bins=edges, density=True, color='C2', alpha=0.45, label='fast', edgecolor = "k", histtype ="stepfilled")

    # Formatting
    ax.set_ylim(0, 0.0030)
    ax.set_xlim(0, 3000)
    ax.set_xlabel('nearest_ref_dist_nm')
    ax.set_ylabel('Density')
    ax.set_title(f'Distance to DnaK tracks ({label[i-1]})')
    ax.legend()

    fig.tight_layout()
    plt.show()
    # To save per dataset:

